# Student AI Impact & Academic Intelligence Analysis

This notebook provides a comprehensive data science analysis of the impact of Generative AI on university students. We explore relationships between AI usage hours, technical literacy, and academic performance (GPA).

**Core Objectives:**
1. **Univariate Analysis:** Examining GPA and AI usage distributions.
2. **Bivariate Analysis:** Investigating the correlation between AI engagement and academic outcomes.
3. **Student Segmentation:** Using K-Means clustering to identify distinct student personas.
4. **Predictive Modeling:** Training a Random Forest model to determine the most influential predictors of GPA in the AI era.
5. **Tool Comparison:** Evaluating the academic performance associated with different AI tools.

### 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
df = pd.read_csv('impact_study.csv')
print(f"Dataset Loaded: {df.shape[0]} records, {df.shape[1]} columns")
df.head()

### 2. Exploratory Data Analysis (EDA)
We start by looking at the distribution of our target variable (GPA) and its relationship with AI usage.

In [ ]:
# GPA Distribution
plt.figure()
df['Semester_GPA'].hist(bins=30, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Distribution of Semester GPA')
plt.xlabel('GPA')
plt.ylabel('Frequency')
plt.show()

# AI Usage vs GPA
plt.figure()
plt.scatter(df['Academic_AI_Hours_Weekly'], df['Semester_GPA'], alpha=0.2, color='blue')
z = np.polyfit(df['Academic_AI_Hours_Weekly'], df['Semester_GPA'], 1)
p = np.poly1d(z)
plt.plot(df['Academic_AI_Hours_Weekly'], p(df['Academic_AI_Hours_Weekly']), "r--", lw=2)
plt.title('Relationship: Academic AI Hours vs Semester GPA')
plt.xlabel('Weekly Academic AI Hours')
plt.ylabel('GPA')
plt.show()

### 3. Student Segmentation (Clustering)
By clustering students based on their behaviors and literacy, we can identify specific groups that may need different support levels.

In [ ]:
# Prepare features for clustering
cluster_features = ['AI_Daily_Hours', 'Academic_AI_Hours_Weekly', 'Semester_GPA', 'Technostress_Overload', 'Literacy_Usage_Skill']
X_cluster = df[cluster_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Apply KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize Clusters
plt.figure()
colors = ['#FF9999', '#66B2FF', '#99FF99']
for i in range(3):
    subset = df[df['Cluster'] == i]
    plt.scatter(subset['Academic_AI_Hours_Weekly'], subset['Semester_GPA'], 
                c=colors[i], label=f'Segment {i}', alpha=0.5)
plt.title('Student Segments: AI Usage vs Academic Performance')
plt.xlabel('Academic AI Hours Weekly')
plt.ylabel('GPA')
plt.legend()
plt.show()

### 4. Predictive Modeling
We use a Random Forest model to predict Semester GPA and extract feature importance to see what truly drives performance.

In [ ]:
features = ['Age', 'Internet_Reliability', 'AI_Daily_Hours', 'Academic_AI_Hours_Weekly', 
            'Paid_Subscription', 'Peer_AI_Consultation', 'Time_Saved_Pct', 
            'Technostress_Overload', 'Literacy_Usage_Skill', 'Hallucination_Detection']

X = df[features]
y = df['Semester_GPA']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure()
plt.barh(range(len(indices)), importances[indices], color='purple', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.title('What Drives GPA? (Feature Importance)')
plt.xlabel('Relative Importance Score')
plt.show()

### 5. Primary AI Tool Efficacy
Finally, we look at which AI tools are associated with higher academic performance.

In [ ]:
tool_impact = df.groupby('Primary_AI_Tool')['Semester_GPA'].mean().sort_values()
plt.figure()
tool_impact.plot(kind='bar', color='teal', alpha=0.7)
plt.ylim(3.0, 3.5)
plt.title('Average GPA by Primary AI Tool')
plt.ylabel('Average GPA')
plt.xticks(rotation=45)
plt.show()